# Coreference Resolution Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: pretrained neural coreference (AllenNLP / spaCy-experimental)

In [ ]:
```python

import spacy

nlp = spacy.load("en_coreference_web_trf")   # experimental model

doc = nlp("Apple announced new products. The company said they would ship soon.")

for cluster in doc._.coref_clusters:

    print(cluster, "->", [m.text for m in cluster])

In [ ]:
```

On a longer document, you get something like:

- Cluster 1: [Apple, The company, they]

- Cluster 2: [new products]

### Step 2: rule-based pronoun resolver (teaching)

See `code/main.py` for a stdlib-only implementation:

1. Extract mentions: named entities (capitalized spans), pronouns (dict lookup), definite descriptions ("the X").

2. For each pronoun, look at the previous K mentions and score them by:

   - gender/number agreement (heuristic)

   - recency (closer wins)

   - syntactic role (subjects preferred)

3. Link the highest-scoring antecedent.

Not competitive with neural models. But it shows the search space and the decisions an end-to-end model must make.

### Step 3: using LLMs for coreference

In [ ]:
```python

prompt = f"""Text: {text}

List every pronoun and noun phrase that refers to a person or company.

Cluster them by what they refer to. Output JSON:

[{{"entity": "Apple", "mentions": ["Apple", "the company", "it"]}}, ...]

"""

In [ ]:
```

Two failure modes to watch. First, LLMs over-merge ("him" and "her" referring to two distinct people). Second, LLMs silently drop mentions in long documents. Always verify with span-offset checks.

### Step 4: evaluation

The standard conll-2012 script computes MUC, B³, CEAF-φ4 and reports the average. For an in-house eval, start with span-level precision and recall on your annotated test set, then add mention-linking F1.

## Exercises

In [ ]:
1. **Easy.** Run the rule-based resolver in `code/main.py` on 5 hand-crafted paragraphs. Measure mention-link accuracy against ground truth.
2. **Medium.** Use a pretrained neural coref model on a news article. Compare clusters against your own manual annotation. Where did it fail?
3. **Hard.** Build a coref-enhanced NER pipeline: NER first, then merge via coref clusters. Measure entity-coverage improvement vs NER-only on 100 articles.